# Model Monitoring — Examples

This notebook demonstrates input quality checks, a simple feature-drift indicator, prediction-distribution monitoring, and quality metrics when labels are available. It uses synthetic data and does not require external downloads.

**Important:** thresholds in this notebook are illustrative. A drift alert is a signal to investigate, not proof that a model is failing.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

## 1. Create reference and simulated current data

In [ ]:
X_array, y = make_classification(
    n_samples=1500,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    random_state=RANDOM_STATE,
)
features = [f"feature_{i}" for i in range(1, 7)]
X_all = pd.DataFrame(X_array, columns=features)
X_train, X_reference, y_train, y_reference = train_test_split(
    X_all, y, test_size=0.35, random_state=RANDOM_STATE, stratify=y
)

# Simulate a production batch with a shift in two features.
rng = np.random.default_rng(RANDOM_STATE)
X_current = X_reference.sample(n=250, random_state=RANDOM_STATE).copy().reset_index(drop=True)
X_current["feature_1"] += rng.normal(loc=0.8, scale=0.25, size=len(X_current))
X_current["feature_3"] += rng.normal(loc=-0.5, scale=0.30, size=len(X_current))

print("Train:", X_train.shape)
print("Reference:", X_reference.shape)
print("Current:", X_current.shape)

## 2. Train a baseline model

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
model.fit(X_train, y_train)
reference_predictions = model.predict(X_reference)
print("Reference accuracy:", round(accuracy_score(y_reference, reference_predictions), 3))

## 3. Check schema and data quality

In [ ]:
def quality_report(df):
    return {
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "missing_rate": df.isna().mean().to_dict(),
        "duplicate_rows": int(df.duplicated().sum()),
        "numeric_summary": df.select_dtypes(include="number").describe().to_dict(),
    }

reference_quality = quality_report(X_reference)
current_quality = quality_report(X_current)

if list(X_reference.columns) != list(X_current.columns):
    raise ValueError("Reference/current schema mismatch")

print("Reference rows:", reference_quality["rows"])
print("Current rows:", current_quality["rows"])
print("Current missing rates:")
print(pd.Series(current_quality["missing_rate"]))

## 4. Calculate a simple numeric drift indicator

In [ ]:
def standardized_mean_shift(reference, current, epsilon=1e-8):
    rows = []
    for feature in reference.select_dtypes(include="number").columns:
        ref = reference[feature].dropna()
        cur = current[feature].dropna()
        if len(ref) == 0 or len(cur) == 0:
            shift = np.nan
        else:
            shift = abs(cur.mean() - ref.mean()) / max(ref.std(ddof=0), epsilon)
        rows.append({
            "feature": feature,
            "reference_mean": ref.mean() if len(ref) else np.nan,
            "current_mean": cur.mean() if len(cur) else np.nan,
            "standardized_mean_shift": shift,
        })
    return pd.DataFrame(rows)

drift_table = standardized_mean_shift(X_reference, X_current)
ILLUSTRATIVE_THRESHOLD = 0.5
drift_table["alert"] = drift_table["standardized_mean_shift"] > ILLUSTRATIVE_THRESHOLD
drift_table.sort_values("standardized_mean_shift", ascending=False)

## 5. Monitor prediction distributions

In [ ]:
current_predictions = model.predict(X_current)

reference_share = pd.Series(reference_predictions).value_counts(normalize=True).sort_index()
current_share = pd.Series(current_predictions).value_counts(normalize=True).sort_index()
prediction_comparison = pd.concat(
    [reference_share.rename("reference_share"),
     current_share.rename("current_share")],
    axis=1,
).fillna(0)
prediction_comparison

## 6. Evaluate quality when labels are available

In [ ]:
def quality_metrics(y_true, predictions):
    return {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
    }

# Demonstration only: labels for the simulated current batch are not real production labels.
# In a real service, calculate metrics only after trustworthy labels arrive.
print("Current production quality: unavailable until labels arrive.")

## 7. Save a monitoring report

In [ ]:
report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "window": "simulated-current-batch",
    "reference_rows": int(len(X_reference)),
    "current_rows": int(len(X_current)),
    "reference_missing_rate": X_reference.isna().mean().to_dict(),
    "current_missing_rate": X_current.isna().mean().to_dict(),
    "feature_drift": drift_table.to_dict(orient="records"),
    "prediction_distribution": prediction_comparison.reset_index().rename(
        columns={"index": "predicted_class"}
    ).to_dict(orient="records"),
    "model_quality": None,
    "notes": "Investigate drift alerts; do not automatically retrain based on this score alone.",
}
report_path = ARTIFACT_DIR / "monitoring_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved report to:", report_path.resolve())

## Key takeaways

- Monitor input quality, service health, predictions, and labeled model quality separately.
- Compare equivalent data windows and include sample counts.
- A simple drift statistic is a diagnostic, not a universal pass/fail test.
- Model quality cannot be known without trustworthy ground-truth labels.
- Investigate upstream causes before rolling back or retraining.